# KND_04 — Step 1: Understand our data

**Purpose:** Open our approved dataset and check its basic details. This notebook does not change the CSV or train a model.

**Our problem:** Predict at arrival whether an animal will stay in the shelter for more than 30 days, so staff can plan care earlier.

**Source:** [Sonoma County Animal Shelter Intake and Outcome](https://data.sonomacounty.ca.gov/Government/Animal-Shelter-Intake-and-Outcome/924a-vesw), California, USA.

We use the saved 2023–2026 extract, downloaded on 15 September 2026. One row is a shelter admission. The same animal may have several separate visits.

## How to run

Open this notebook in Jupyter, VS Code or Google Colab. Keep the CSV beside the notebook, or upload both files to Colab. Run the cells from top to bottom.

For the viva, each member should explain what each check tells us. These first checks are only the beginning: EDA, target construction, splitting and preprocessing still need to be completed.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

filename = "Sonoma_Animal_Shelter_2023-2026_RAW_9940.csv"
candidates = [Path(filename), Path("outputs") / filename, Path("/content") / filename]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place our CSV beside this notebook, or upload it to Colab first.")

df = pd.read_csv(DATA_PATH, dtype={"id": "string", "impound_number": "string"})
print("Rows:", len(df))
print("Columns:", len(df.columns))
expected_hash = "fd465b0dc5178bd13d0b9e70959320137f9526e635fe982c5ffc2c115faf61e0"
print("Matches our saved proposal dataset:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == expected_hash)
print(df.head(3).to_string(index=False))

Rows: 9940
Columns: 24
Matches our saved proposal dataset: True
     name type          breed          color    sex  size           date_of_birth impound_number kennel_number      id             intake_date            outcome_date  days_in_shelter intake_type intake_subtype    outcome_type         outcome_subtype intake_condition outcome_condition intake_jurisdiction outcome_jurisdiction  zip_code                               location  intake_total
    LISTO  DOG GREAT PYRENEES          WHITE   Male LARGE 2021-01-15T00:00:00.000     K23-043481          DS63 A414670 2023-01-01T00:00:00.000 2023-01-03T00:00:00.000                2       STRAY          FIELD RETURN TO OWNER    OVER THE COUNTER_WEB          UNKNOWN           PENDING              COUNTY               COUNTY   95407.0 \n,  95407\n(38.4127094, -122.7412153)             1
    ANGUS  DOG  BORDER COLLIE    BLACK/WHITE   Male   MED 2019-10-01T00:00:00.000     K23-043482         DQ112 A414672 2023-01-01T00:00:00.000 2023-01-21T00

## 1. What does each column contain?

Read the names and examples together. A date can initially load as text. We will convert dates carefully during preprocessing. Use the source data dictionary to confirm meanings; do not guess from column names alone.

In [2]:
print(df.dtypes.to_string())
print("\nAnimal types:")
print(df["type"].value_counts(dropna=False).to_string())

name                        str
type                        str
breed                       str
color                       str
sex                         str
size                        str
date_of_birth               str
impound_number           string
kennel_number               str
id                       string
intake_date                 str
outcome_date                str
days_in_shelter           int64
intake_type                 str
intake_subtype              str
outcome_type                str
outcome_subtype             str
intake_condition            str
outcome_condition           str
intake_jurisdiction         str
outcome_jurisdiction        str
zip_code                float64
location                    str
intake_total              int64

Animal types:
type
DOG      5716
CAT      3262
OTHER     962


## 2. Which information is missing?

A blank date of birth means we cannot directly calculate that animal's age. It does not automatically mean we should delete the whole row.

An entry such as `UNKNOWN` is also important, even though Python does not count it as a blank cell.

In [3]:
missing = df.isna().sum().sort_values(ascending=False)
print("Blank values by column:")
print(missing[missing > 0].to_string())
print("\nCommon values in sex and intake condition:")
for column in ["sex", "intake_condition"]:
    print("\n" + column)
    print(df[column].value_counts(dropna=False).head(10).to_string())

Blank values by column:
name                    2644
date_of_birth           2151
outcome_jurisdiction    1718
zip_code                1612
location                1612
outcome_subtype          261
outcome_condition        242
outcome_type             212
outcome_date             211
kennel_number              3
size                       1

Common values in sex and intake condition:

sex
sex
Male        2445
Neutered    2403
Female      2103
Spayed      2072
Unknown      917

intake_condition
intake_condition
UNKNOWN                 7433
HEALTHY                 2006
UNTREATABLE              201
TREATABLE/REHAB          187
TREATABLE/MANAGEABLE     113


## 3. Are records repeated?

**Exact duplicate:** Every column is the same.

**Repeated animal ID:** Could be a real return visit, such as a dog arriving in January and returning in July. We must not remove every repeated ID.

**Same animal and arrival date:** A candidate for closer review. This check alone does not prove the record is wrong.

In [4]:
print("Extra exact duplicate rows:", df.duplicated().sum())
same_admission = df.duplicated(subset=["id", "intake_date"], keep=False)
print("Rows sharing an animal ID and intake date:", same_admission.sum())
print("These rows need review before any removal.")

Extra exact duplicate rows: 0
Rows sharing an animal ID and intake date: 24
These rows need review before any removal.


## 4. Do the dates make sense?

For example, an animal cannot be born after it arrives. We will treat impossible birth dates as missing information, rather than treating the resulting negative age as real.

In [5]:
intake = pd.to_datetime(df["intake_date"], errors="coerce")
birth = pd.to_datetime(df["date_of_birth"], errors="coerce")
print("Earliest intake:", intake.min())
print("Latest intake:", intake.max())
print("Nonblank intake dates that could not be read:", (df["intake_date"].notna() & intake.isna()).sum())
print("Nonblank birth dates that could not be read:", (df["date_of_birth"].notna() & birth.isna()).sum())
print("Birth dates after intake:", (birth > intake).sum())

Earliest intake: 2023-01-01 00:00:00
Latest intake: 2026-09-14 00:00:00
Nonblank intake dates that could not be read: 0
Nonblank birth dates that could not be read: 0
Birth dates after intake: 15


## Group discussion — fill this in yourselves

1. What does one row represent?
2. Why is an animal ID repeated sometimes?
3. Why should we avoid deleting every row with a missing birth date?
4. Why can we not use `days_in_shelter` as a prediction input?
5. What did each member run, check and explain?

**Our answers:**

- Add your own explanations here.

## Next step

Create the target and decide which records have enough follow-up. A completed 12-day stay gets label 0; a 45-day stay gets label 1. An animal still present after only 10 days has no known answer yet.

Use our fixed observation date, 14 September 2026. Do not use today's date to label this saved extract. Keep a count and reason for each excluded row. The proposal's initial screening retained 9,670 records; reproduce and review this before calling preprocessing complete.

Before choosing transformations, reserve the future test period. Fit replacement ages, category encoding and scaling using training data only. For example, do not calculate the replacement age using animals in the test set.

**Step 1 is complete when:** every member can run this notebook and explain the missing-value, duplicate and date checks. Model training and the application come later.